# Human Brain Segmentation — Track A: Allen Human SectionImage SVGs

Fine-tune DINOv2-Large + UperNet on Allen Human Brain Atlas annotated sections.
Uses the same model architecture and training recipe as the mouse Run 9 (final)
model, adapted for RGB images with sparse SVG polygon annotations.

**Data source:** 4,463 annotated SVG + JPEG pairs across 6 donors (Nissl stain).

**Key differences from mouse training:**
- RGB images (not grayscale) — native 3-channel normalization
- Sparse annotations — unlabeled pixels = 255 (ignored by CE loss)
- Donor-based split (not spatial/interleaved) — tests cross-individual generalization
- Pre-cached dataset (resized to 1024px + rasterized masks as .npz) for fast loading
- Graph 10 ontology with present_mapping (~597 observed structure IDs)

## Config (Run 9 recipe, 200 epochs)
- DINOv2-Large + UperNet, ~597 classes (present structures only)
- Last 4 backbone blocks unfrozen (20-23)
- Differential LR: backbone 1e-5, head 1e-4
- Baseline augmentation: flip + rot15 + jitter (mask fill=255)
- CE loss with ignore_index=255
- Batch 2 x 2 grad accum = effective batch 4
- 200 epochs (same as mouse Run 9 and BigBrain)
- Pre-cached images at 1024px max dimension

In [0]:
# Cell 0 — Install project wheel from DBFS

%pip install /dbfs/FileStore/wheels/histological_image_analysis-0.1.0-py3-none-any.whl
dbutils.library.restartPython()

Processing /dbfs/FileStore/wheels/histological_image_analysis-0.1.0-py3-none-any.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 10.8 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.5.2
    Not uninstalling accelerate at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-bb10ba06-8750-48be-89ba-f27c056cc22f
    Can't uninstall 'accelerate'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Cell 1 — Configuration

import os
import mlflow

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"] = "true"

EXPERIMENT_NAME = "/Users/noel.nosse@grainger.com/histology-human-brain-segmentation"
try:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
except Exception:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
mlflow.set_experiment(experiment_id=experiment_id)
print(f"MLflow experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")

# ---------- Databricks paths ----------
WORKSPACE_BASE = "/Workspace/Users/noel.nosse@grainger.com/visual-model-ft/histology"
ONTOLOGY_PATH = f"{WORKSPACE_BASE}/ontology/structure_graph_10.json"
METADATA_PATH = f"{WORKSPACE_BASE}/metadata/human_atlas_images_metadata.json"
IMAGES_DIR = f"{WORKSPACE_BASE}/human_atlas/images"
SVGS_DIR = f"{WORKSPACE_BASE}/human_atlas/svgs"

# ---------- JFrog / HuggingFace model ----------
HF_ENDPOINT = os.environ.get(
    "HF_ENDPOINT",
    "https://graingerreadonly.jfrog.io/artifactory/api/huggingfaceml/huggingfaceml-remote",
)
MODEL_REPO_ID = "facebook/dinov2-large"
MODEL_CACHE_DIR = "/tmp/dinov2-large"

# ---------- Donor split ----------
TRAIN_DONORS = ["H0351.2002", "H0351.2001", "H0351.1012", "H0351.1009"]
VAL_DONORS = ["H0351.1016"]
TEST_DONORS = ["H0351.1015"]

# ---------- Training hyperparameters (Run 9 recipe) ----------
BACKBONE_LR = 1e-5
HEAD_LR = 1e-4
NUM_UNFROZEN_BLOCKS = 4
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 2
CROP_SIZE = 518
NUM_EPOCHS = 200

OUTPUT_DIR = "/tmp/checkpoints/human-allen"
FINAL_MODEL_DIR = "/dbfs/FileStore/allen_brain_data/models/human-allen"

# ---------- Pre-cache config ----------
CACHE_DIR = "/tmp/allen_cache"
CACHE_MAX_DIM = 1024  # Resize images to max 1024px before caching

HYPERPARAMS = {
    "track": "A (Allen Human SVGs)",
    "crop_size": CROP_SIZE,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "backbone_lr": BACKBONE_LR,
    "head_lr": HEAD_LR,
    "num_unfrozen_blocks": NUM_UNFROZEN_BLOCKS,
    "num_epochs": NUM_EPOCHS,
    "freeze_backbone": False,
    "gradient_checkpointing": "backbone_only_non_reentrant",
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "model": MODEL_REPO_ID,
    "dataset": "Allen Human SectionImage SVGs (Graph 10)",
    "augmentation": "baseline: flip_50pct, rot15_always, jitter_always (mask fill=255)",
    "loss": "CE(ignore_index=255)",
    "split_strategy": "by_donor",
    "train_donors": ", ".join(TRAIN_DONORS),
    "val_donors": ", ".join(VAL_DONORS),
    "test_donors": ", ".join(TEST_DONORS),
    "cache_max_dim": CACHE_MAX_DIM,
    "run_description": "Track A: Allen Human SVGs, donor-split, present_mapping, 200ep, pre-cached 1024px",
}

print(f"{'='*65}")
print(f"TRACK A: ALLEN HUMAN BRAIN — SVG ANNOTATIONS")
print(f"{'='*65}")
print(f"Ontology:        Graph 10")
print(f"Batch size:      {BATCH_SIZE} (x{GRADIENT_ACCUMULATION_STEPS} = effective {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS})")
print(f"Backbone LR:     {BACKBONE_LR}")
print(f"Head LR:         {HEAD_LR}")
print(f"Unfrozen blocks: last {NUM_UNFROZEN_BLOCKS} (blocks {24 - NUM_UNFROZEN_BLOCKS}-23)")
print(f"Epochs:          {NUM_EPOCHS}")
print(f"Cache:           {CACHE_DIR} (max {CACHE_MAX_DIM}px)")
print(f"Train donors:    {TRAIN_DONORS}")
print(f"Val donors:      {VAL_DONORS}")
print(f"Test donors:     {TEST_DONORS}")
print(f"Output dir:      {FINAL_MODEL_DIR}")

MLflow experiment: /Users/noel.nosse@grainger.com/histology-human-brain-segmentation (ID: 4022263052856815)
TRACK A: ALLEN HUMAN BRAIN — SVG ANNOTATIONS
Ontology:        Graph 10
Batch size:      2 (x2 = effective 4)
Backbone LR:     1e-05
Head LR:         0.0001
Unfrozen blocks: last 4 (blocks 20-23)
Epochs:          200
Cache:           /tmp/allen_cache (max 1024px)
Train donors:    ['H0351.2002', 'H0351.2001', 'H0351.1012', 'H0351.1009']
Val donors:      ['H0351.1016']
Test donors:     ['H0351.1015']
Output dir:      /dbfs/FileStore/allen_brain_data/models/human-allen


In [0]:
# Cell 2 — Download model weights from JFrog Artifactory mirror

import time
from huggingface_hub import snapshot_download

print(f"Downloading {MODEL_REPO_ID} from Artifactory...")
for attempt in range(1, 4):
    try:
        model_path = snapshot_download(
            repo_id=MODEL_REPO_ID,
            endpoint=HF_ENDPOINT,
            etag_timeout=86400,
            cache_dir=MODEL_CACHE_DIR,
        )
        break
    except Exception as e:
        if attempt < 3:
            wait = 2 ** attempt
            print(f"  Attempt {attempt} failed ({e.__class__.__name__}), retrying in {wait}s...")
            time.sleep(wait)
        else:
            raise

print(f"Model downloaded to: {model_path}")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

(…)d21ea90829a6652ed6f75ec47/.gitattributes:   0%|          | 0.00/1.52k [00:00<?, ?B/s]

(…)6e6d21ea90829a6652ed6f75ec47/config.json:   0%|          | 0.00/549 [00:00<?, ?B/s]

(…)496e6d21ea90829a6652ed6f75ec47/README.md:   0%|          | 0.00/3.03k [00:00<?, ?B/s]

(…)a6652ed6f75ec47/preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

(…)ea90829a6652ed6f75ec47/pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

(…)ea90829a6652ed6f75ec47/model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Model downloaded to: /tmp/dinov2-large/models--facebook--dinov2-large/snapshots/a18992645e496e6d21ea90829a6652ed6f75ec47


In [0]:
# Cell 3 — Build Allen Human datasets (with pre-cache)
#
# 1. Load metadata JSON to find annotated image/SVG pairs
# 2. Split by donor (4 train / 1 val / 1 test)
# 3. Scan training SVGs for unique structure IDs → build present_mapping
# 4. Pre-cache: resize images to CACHE_MAX_DIM + rasterize SVGs → .npz
# 5. Create AllenHumanDataset instances with cache_dir

import json
from pathlib import Path
from xml.etree import ElementTree as ET

from histological_image_analysis.ontology import OntologyMapper
from histological_image_analysis.svg_rasterizer import SVGRasterizer
from histological_image_analysis.dataset import AllenHumanDataset, split_by_donor

# --- Step 1: Load metadata and build (image, svg, donor) triples ---
with open(METADATA_PATH) as f:
    metadata = json.load(f)

print(f"Total metadata entries: {len(metadata)}")

# Build triples: (image_path, svg_path, donor_id)
# Metadata keys: "_donor", "section_number", "id"
# Filename pattern: {DONOR}_{SECTION:04d}_{ID}.{ext}
all_pairs = []
missing_images = 0
missing_svgs = 0

for entry in metadata:
    if not entry.get("annotated", False):
        continue

    donor_id = entry["_donor"]
    section_number = entry["section_number"]
    image_id = entry["id"]

    filename_base = f"{donor_id}_{section_number:04d}_{image_id}"
    image_path = Path(IMAGES_DIR) / f"{filename_base}.jpg"
    svg_path = Path(SVGS_DIR) / f"{filename_base}.svg"

    if not image_path.exists():
        missing_images += 1
        continue
    if not svg_path.exists():
        missing_svgs += 1
        continue

    all_pairs.append((image_path, svg_path, donor_id))

print(f"Annotated pairs found: {len(all_pairs)}")
if missing_images > 0:
    print(f"  Missing images: {missing_images}")
if missing_svgs > 0:
    print(f"  Missing SVGs: {missing_svgs}")

# --- Step 2: Split by donor ---
splits = split_by_donor(all_pairs, TRAIN_DONORS, VAL_DONORS, TEST_DONORS)
print(f"Train: {len(splits['train'])} | Val: {len(splits['val'])} | Test: {len(splits['test'])}")

# --- Step 3: Scan training SVGs for present structure IDs ---
print("Scanning training SVGs for structure IDs...")
all_structure_ids = set()
for img_path, svg_path in splits["train"]:
    tree = ET.parse(str(svg_path))
    root = tree.getroot()
    for elem in root.iter():
        sid_str = elem.get("structure_id", "")
        if sid_str:
            try:
                all_structure_ids.add(int(sid_str))
            except ValueError:
                pass

print(f"Unique structure IDs in training SVGs: {len(all_structure_ids)}")

# Build mappings
mapper = OntologyMapper(ONTOLOGY_PATH)
full_mapping = mapper.build_full_mapping()

# present_class_ids = class IDs from full_mapping that correspond to observed structure IDs
present_class_ids = {
    full_mapping[sid]
    for sid in all_structure_ids
    if sid in full_mapping
}
present_mapping = mapper.build_present_mapping(full_mapping, present_class_ids)
NUM_LABELS = mapper.get_num_labels(present_mapping)
class_names = mapper.get_class_names(present_mapping)

HYPERPARAMS["num_labels"] = NUM_LABELS
HYPERPARAMS["unique_structure_ids"] = len(all_structure_ids)
print(f"Present mapping: {NUM_LABELS} classes")

# --- Step 4: Pre-cache all splits ---
rasterizer = SVGRasterizer(mapper)

import time
all_split_pairs = splits["train"] + splits["val"] + splits["test"]
print(f"\nPre-caching {len(all_split_pairs)} images to {CACHE_DIR} (max {CACHE_MAX_DIM}px)...")
cache_start = time.time()
cached_count = AllenHumanDataset.build_cache(
    image_svg_pairs=all_split_pairs,
    rasterizer=rasterizer,
    mapping=present_mapping,
    cache_dir=CACHE_DIR,
    max_dim=CACHE_MAX_DIM,
)
cache_elapsed = time.time() - cache_start
print(f"Cached {cached_count} files in {cache_elapsed:.0f}s ({cache_elapsed/60:.1f} min)")

# --- Step 5: Create datasets with cache ---
train_ds = AllenHumanDataset(
    image_svg_pairs=splits["train"],
    rasterizer=rasterizer,
    mapping=present_mapping,
    crop_size=CROP_SIZE,
    augment=True,
    cache_dir=CACHE_DIR,
)
val_ds = AllenHumanDataset(
    image_svg_pairs=splits["val"],
    rasterizer=rasterizer,
    mapping=present_mapping,
    crop_size=CROP_SIZE,
    augment=False,
    cache_dir=CACHE_DIR,
)

HYPERPARAMS["train_samples"] = len(train_ds)
HYPERPARAMS["val_samples"] = len(val_ds)
print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

# Verify a sample
sample = train_ds[0]
print(f"pixel_values: {sample['pixel_values'].shape}, labels: {sample['labels'].shape}")
unique_labels = set(sample['labels'].numpy().ravel())
print(f"Unique labels in sample: {sorted(unique_labels)}")

Total metadata entries: 14566
Annotated pairs found: 4463
Train: 3188 | Val: 641 | Test: 634
Scanning training SVGs for structure IDs...
Unique structure IDs in training SVGs: 596
Present mapping: 597 classes

Pre-caching 4463 images to /tmp/allen_cache (max 1024px)...
Cached 4463 files in 28239s (470.7 min)
Train samples: 3188 | Val samples: 641
pixel_values: torch.Size([3, 518, 518]), labels: torch.Size([518, 518])
Unique labels in sample: [np.int64(255), np.int64(376), np.int64(377)]


In [0]:
# Cell 4 — Create model with partially unfrozen backbone

import torch
from histological_image_analysis.training import create_model

model = create_model(
    num_labels=NUM_LABELS,
    freeze_backbone=False,
    pretrained_backbone_path=model_path,
)

first_frozen_block = 24 - NUM_UNFROZEN_BLOCKS

for param in model.backbone.embeddings.parameters():
    param.requires_grad = False
for i in range(first_frozen_block):
    for param in model.backbone.encoder.layer[i].parameters():
        param.requires_grad = False

model.backbone.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
print("Gradient checkpointing: ENABLED (backbone only, use_reentrant=False)")

backbone_frozen = sum(p.numel() for p in model.backbone.parameters() if not p.requires_grad)
backbone_trainable = sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
head_trainable = sum(
    p.numel() for n, p in model.named_parameters()
    if p.requires_grad and "backbone" not in n
)
total = sum(p.numel() for p in model.parameters())

print(f"Backbone frozen:    {backbone_frozen:>12,} params")
print(f"Backbone trainable: {backbone_trainable:>12,} params (blocks {first_frozen_block}-23 + layernorm)")
print(f"Head trainable:     {head_trainable:>12,} params")
print(f"Total trainable:    {backbone_trainable + head_trainable:>12,} / {total:,} ({(backbone_trainable + head_trainable) / total * 100:.1f}%)")

model.eval()
with torch.no_grad():
    dummy = torch.randn(1, 3, CROP_SIZE, CROP_SIZE)
    if torch.cuda.is_available():
        model = model.cuda()
        dummy = dummy.cuda()
    out = model(pixel_values=dummy)
    print(f"\nLogits shape: {out.logits.shape}")
    assert out.logits.shape == (1, NUM_LABELS, CROP_SIZE, CROP_SIZE)

del out, dummy
torch.cuda.empty_cache()
model = model.cpu()
torch.cuda.empty_cache()
print("Forward pass OK")

2026-03-20 08:08:37.029176: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773994117.040843    2670 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773994117.044460    2670 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773994117.054989    2670 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773994117.054998    2670 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773994117.055001    2670 computation_placer.cc:177] computation placer alr

[2026-03-20 08:08:43,481] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


df: /root/.triton/autotune: No such file or directory
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


Gradient checkpointing: ENABLED (backbone only, use_reentrant=False)
Backbone frozen:     253,973,504 params
Backbone trainable:   50,395,136 params (blocks 20-23 + layernorm)
Head trainable:       37,172,650 params
Total trainable:      87,567,786 / 341,541,290 (25.6%)

Logits shape: torch.Size([1, 597, 518, 518])
Forward pass OK


In [0]:
# Cell 5 — Train with differential learning rate (200 epochs)

import torch
from datetime import datetime
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from histological_image_analysis.training import (
    get_training_args,
    make_compute_metrics,
    preprocess_logits_for_metrics,
)
from transformers import Trainer

run_name = f"human-allen-{NUM_LABELS}class-{datetime.now().strftime('%Y%m%d-%H%M')}"
mlflow.start_run(run_name=run_name)
mlflow.log_params(HYPERPARAMS)

training_args = get_training_args(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=HEAD_LR,
    fp16=torch.cuda.is_available(),
    report_to="mlflow",
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    ddp_find_unused_parameters=True,
    dataloader_drop_last=True,
)

backbone_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and "backbone" in n
]
head_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and "backbone" not in n
]

optimizer = AdamW(
    [
        {"params": backbone_params, "lr": BACKBONE_LR},
        {"params": head_params, "lr": HEAD_LR},
    ],
    weight_decay=HYPERPARAMS["weight_decay"],
)

num_training_steps = (len(train_ds) // (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)) * NUM_EPOCHS
num_warmup_steps = int(num_training_steps * HYPERPARAMS["warmup_ratio"])

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print(f"Run: {run_name}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Total steps: {num_training_steps} | Warmup: {num_warmup_steps}")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=make_compute_metrics(NUM_LABELS),
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    optimizers=(optimizer, scheduler),
)

trainer.train()

2026/03/20 08:08:53 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/03/20 08:08:53 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Run: human-allen-597class-20260320-0808
Epochs: 200
Total steps: 159400 | Warmup: 15940


Trainer is attempting to log a value of "{0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2', 3: 'LABEL_3', 4: 'LABEL_4', 5: 'LABEL_5', 6: 'LABEL_6', 7: 'LABEL_7', 8: 'LABEL_8', 9: 'LABEL_9', 10: 'LABEL_10', 11: 'LABEL_11', 12: 'LABEL_12', 13: 'LABEL_13', 14: 'LABEL_14', 15: 'LABEL_15', 16: 'LABEL_16', 17: 'LABEL_17', 18: 'LABEL_18', 19: 'LABEL_19', 20: 'LABEL_20', 21: 'LABEL_21', 22: 'LABEL_22', 23: 'LABEL_23', 24: 'LABEL_24', 25: 'LABEL_25', 26: 'LABEL_26', 27: 'LABEL_27', 28: 'LABEL_28', 29: 'LABEL_29', 30: 'LABEL_30', 31: 'LABEL_31', 32: 'LABEL_32', 33: 'LABEL_33', 34: 'LABEL_34', 35: 'LABEL_35', 36: 'LABEL_36', 37: 'LABEL_37', 38: 'LABEL_38', 39: 'LABEL_39', 40: 'LABEL_40', 41: 'LABEL_41', 42: 'LABEL_42', 43: 'LABEL_43', 44: 'LABEL_44', 45: 'LABEL_45', 46: 'LABEL_46', 47: 'LABEL_47', 48: 'LABEL_48', 49: 'LABEL_49', 50: 'LABEL_50', 51: 'LABEL_51', 52: 'LABEL_52', 53: 'LABEL_53', 54: 'LABEL_54', 55: 'LABEL_55', 56: 'LABEL_56', 57: 'LABEL_57', 58: 'LABEL_58', 59: 'LABEL_59', 60: 'LABEL_60', 61

Epoch,Training Loss,Validation Loss,Mean Iou,Overall Accuracy,Iou Class 0,Iou Class 1,Iou Class 2,Iou Class 3,Iou Class 4,Iou Class 5,Iou Class 6,Iou Class 7,Iou Class 8,Iou Class 9,Iou Class 10,Iou Class 11,Iou Class 12,Iou Class 13,Iou Class 14,Iou Class 15,Iou Class 16,Iou Class 17,Iou Class 18,Iou Class 19,Iou Class 20,Iou Class 21,Iou Class 22,Iou Class 23,Iou Class 24,Iou Class 25,Iou Class 26,Iou Class 27,Iou Class 28,Iou Class 29,Iou Class 30,Iou Class 31,Iou Class 32,Iou Class 33,Iou Class 34,Iou Class 35,Iou Class 36,Iou Class 37,Iou Class 38,Iou Class 39,Iou Class 40,Iou Class 41,Iou Class 42,Iou Class 43,Iou Class 44,Iou Class 45,Iou Class 46,Iou Class 47,Iou Class 48,Iou Class 49,Iou Class 50,Iou Class 51,Iou Class 52,Iou Class 53,Iou Class 54,Iou Class 55,Iou Class 56,Iou Class 57,Iou Class 58,Iou Class 59,Iou Class 60,Iou Class 61,Iou Class 62,Iou Class 63,Iou Class 64,Iou Class 65,Iou Class 66,Iou Class 67,Iou Class 68,Iou Class 69,Iou Class 70,Iou Class 71,Iou Class 72,Iou Class 73,Iou Class 74,Iou Class 75,Iou Class 76,Iou Class 77,Iou Class 78,Iou Class 79,Iou Class 80,Iou Class 81,Iou Class 82,Iou Class 83,Iou Class 84,Iou Class 85,Iou Class 86,Iou Class 87,Iou Class 88,Iou Class 89,Iou Class 90,Iou Class 91,Iou Class 92,Iou Class 93,Iou Class 94,Iou Class 95,Iou Class 96,Iou Class 97,Iou Class 98,Iou Class 99,Iou Class 100,Iou Class 101,Iou Class 102,Iou Class 103,Iou Class 104,Iou Class 105,Iou Class 106,Iou Class 107,Iou Class 108,Iou Class 109,Iou Class 110,Iou Class 111,Iou Class 112,Iou Class 113,Iou Class 114,Iou Class 115,Iou Class 116,Iou Class 117,Iou Class 118,Iou Class 119,Iou Class 120,Iou Class 121,Iou Class 122,Iou Class 123,Iou Class 124,Iou Class 125,Iou Class 126,Iou Class 127,Iou Class 128,Iou Class 129,Iou Class 130,Iou Class 131,Iou Class 132,Iou Class 133,Iou Class 134,Iou Class 135,Iou Class 136,Iou Class 137,Iou Class 138,Iou Class 139,Iou Class 140,Iou Class 141,Iou Class 142,Iou Class 143,Iou Class 144,Iou Class 145,Iou Class 146,Iou Class 147,Iou Class 148,Iou Class 149,Iou Class 150,Iou Class 151,Iou Class 152,Iou Class 153,Iou Class 154,Iou Class 155,Iou Class 156,Iou Class 157,Iou Class 158,Iou Class 159,Iou Class 160,Iou Class 161,Iou Class 162,Iou Class 163,Iou Class 164,Iou Class 165,Iou Class 166,Iou Class 167,Iou Class 168,Iou Class 169,Iou Class 170,Iou Class 171,Iou Class 172,Iou Class 173,Iou Class 174,Iou Class 175,Iou Class 176,Iou Class 177,Iou Class 178,Iou Class 179,Iou Class 180,Iou Class 181,Iou Class 182,Iou Class 183,Iou Class 184,Iou Class 185,Iou Class 186,Iou Class 187,Iou Class 188,Iou Class 189,Iou Class 190,Iou Class 191,Iou Class 192,Iou Class 193,Iou Class 194,Iou Class 195,Iou Class 196,Iou Class 197,Iou Class 198,Iou Class 199,Iou Class 200,Iou Class 201,Iou Class 202,Iou Class 203,Iou Class 204,Iou Class 205,Iou Class 206,Iou Class 207,Iou Class 208,Iou Class 209,Iou Class 210,Iou Class 211,Iou Class 212,Iou Class 213,Iou Class 214,Iou Class 215,Iou Class 216,Iou Class 217,Iou Class 218,Iou Class 219,Iou Class 220,Iou Class 221,Iou Class 222,Iou Class 223,Iou Class 224,Iou Class 225,Iou Class 226,Iou Class 227,Iou Class 228,Iou Class 229,Iou Class 230,Iou Class 231,Iou Class 232,Iou Class 233,Iou Class 234,Iou Class 235,Iou Class 236,Iou Class 237,Iou Class 238,Iou Class 239,Iou Class 240,Iou Class 241,Iou Class 242,Iou Class 243,Iou Class 244,Iou Class 245,Iou Class 246,Iou Class 247,Iou Class 248,Iou Class 249,Iou Class 250,Iou Class 251,Iou Class 252,Iou Class 253,Iou Class 254,Iou Class 255,Iou Class 256,Iou Class 257,Iou Class 258,Iou Class 259,Iou Class 260,Iou Class 261,Iou Class 262,Iou Class 263,Iou Class 264,Iou Class 265,Iou Class 266,Iou Class 267,Iou Class 268,Iou Class 269,Iou Class 270,Iou Class 271,Iou Class 272,Iou Class 273,Iou Class 274,Iou Class 275,Iou Class 276,Iou Class 277,Iou Class 278,Iou Class 279,Iou Class 280,Iou Class 281,Iou Class 282,Iou Class 283,Iou Class 284,Iou Class 285,Iou Class 286,Iou Class 287,Iou Class 288,Io

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
ConnectionResetError: [Errno 104] Connection reset by peer

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/databricks/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/databricks/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/clientserver.py", line 566, in send_command
    raise Py4JNetworkError(
py4j.protocol.Py4JNetworkError: Error while sending or receiving


In [0]:
# Cell 6 — Evaluation: center-crop + sliding window
#
# Center-crop eval uses the Trainer (same as mouse).
# Sliding window eval tiles the full-resolution image with overlapping
# 518x518 windows and averages logits in overlap regions.
# Key difference from mouse: RGB normalization (not grayscale replication).
#
# NOTE: Allen images at downsample=4 are ~6K-12K pixels per axis.
# Sliding window logit buffer = num_labels * H * W * 4 bytes.
# At native size: ~524 * 8560 * 6796 * 4 ≈ 122 GB — will OOM.
# Fix: resize to max dimension SW_MAX_DIM before tiling.

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

# --- Center-crop evaluation ---
eval_results = trainer.evaluate()
print("Center-crop evaluation results:")
for k, v in sorted(eval_results.items()):
    if isinstance(v, float) and not k.startswith("eval_iou_class_"):
        print(f"  {k}: {v:.4f}")

current_miou = eval_results.get('eval_mean_iou', 0.0)
current_acc = eval_results.get('eval_overall_accuracy', 0.0)

mlflow.log_metrics({
    "cc_mean_iou": current_miou,
    "cc_overall_accuracy": current_acc,
})

# Per-class IoU summary
class_ious = {}
for cls in range(NUM_LABELS):
    iou_key = f"eval_iou_class_{cls}"
    iou = eval_results.get(iou_key, float('nan'))
    if not np.isnan(iou):
        class_ious[cls] = iou

sorted_ious = sorted(class_ious.items(), key=lambda x: x[1], reverse=True)
print(f"\nTop 10 classes by IoU:")
for cls, iou in sorted_ious[:10]:
    name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
    print(f"  {cls:4d} ({name:40s}): {iou:.4f}")

print(f"\nBottom 10 classes by IoU (non-zero):")
nonzero_ious = [(c, i) for c, i in sorted_ious if i > 0]
for cls, iou in nonzero_ious[-10:]:
    name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
    print(f"  {cls:4d} ({name:40s}): {iou:.4f}")

print(f"\nClasses with valid IoU: {len(class_ious)} / {NUM_LABELS}")

# --- Sliding window evaluation ---
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
STRIDE = CROP_SIZE // 2  # 259 — 50% overlap
SW_MAX_DIM = 1024  # Resize images to this max dimension before tiling


def normalize_tile_rgb(tile):
    """uint8 RGB (H, W, 3) -> float32 tensor (1, 3, H, W)."""
    img = tile.astype(np.float32) / 255.0
    img_3ch = np.transpose(img, (2, 0, 1))  # (3, H, W)
    for c in range(3):
        img_3ch[c] = (img_3ch[c] - IMAGENET_MEAN[c]) / IMAGENET_STD[c]
    return torch.from_numpy(img_3ch).unsqueeze(0)


def sliding_window_predict_rgb(model, image, num_labels, crop_size, stride, device):
    """Predict full-resolution segmentation using overlapping tiles.

    Args:
        image: uint8 RGB (H, W, 3)
        Returns: int64 prediction map (H, W)
    """
    h, w = image.shape[:2]
    pad_h = max(0, crop_size - h)
    pad_w = max(0, crop_size - w)
    if pad_h > 0 or pad_w > 0:
        image = np.pad(image,
                       ((0, pad_h), (0, pad_w), (0, 0)),
                       mode='constant', constant_values=0)
    ph, pw = image.shape[:2]

    logit_sum = np.zeros((num_labels, ph, pw), dtype=np.float32)
    count_map = np.zeros((ph, pw), dtype=np.float32)

    y_starts = list(range(0, ph - crop_size + 1, stride))
    x_starts = list(range(0, pw - crop_size + 1, stride))
    if y_starts[-1] + crop_size < ph:
        y_starts.append(ph - crop_size)
    if x_starts[-1] + crop_size < pw:
        x_starts.append(pw - crop_size)

    use_cuda = device.type == "cuda"
    model.eval()
    for y in y_starts:
        for x in x_starts:
            tile = image[y:y + crop_size, x:x + crop_size]  # (H, W, 3)
            pixel_values = normalize_tile_rgb(tile).to(device)
            with torch.no_grad():
                if use_cuda:
                    with torch.amp.autocast("cuda", dtype=torch.float16):
                        logits = model(pixel_values=pixel_values).logits
                else:
                    logits = model(pixel_values=pixel_values).logits
            tile_logits = logits.squeeze(0).float().cpu().numpy()
            logit_sum[:, y:y + crop_size, x:x + crop_size] += tile_logits
            count_map[y:y + crop_size, x:x + crop_size] += 1.0

    count_map = np.maximum(count_map, 1.0)
    avg_logits = logit_sum / count_map[np.newaxis, :, :]
    pred = avg_logits.argmax(axis=0).astype(np.int64)
    return pred[:h, :w]


def resize_for_sliding_window(img_pil, mask, max_dim):
    """Resize image and mask so the longest side is max_dim pixels.

    Returns (resized_image_np, resized_mask_np, scale_factor).
    If already small enough, returns originals unchanged.
    """
    w, h = img_pil.size
    longest = max(h, w)
    if longest <= max_dim:
        return np.array(img_pil), mask, 1.0

    scale = max_dim / longest
    new_w = int(w * scale)
    new_h = int(h * scale)

    resized_img = img_pil.resize((new_w, new_h), resample=Image.BILINEAR)
    resized_mask = Image.fromarray(mask.astype(np.int32)).resize(
        (new_w, new_h), resample=Image.NEAREST
    )
    return np.array(resized_img), np.array(resized_mask).astype(np.int64), scale


# Run sliding window eval on a subset of val images
# (full val set may be slow with lazy loading + tiled inference)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

SW_MAX_IMAGES = 50  # Limit for sliding window eval (full val=641 would be very slow)
val_pairs_for_sw = splits["val"][:SW_MAX_IMAGES]

all_sw_preds = []
all_sw_labels = []

print(f"\nSliding window eval: {len(val_pairs_for_sw)} images, crop={CROP_SIZE}, stride={STRIDE}, max_dim={SW_MAX_DIM}")

for img_path, svg_path in tqdm(val_pairs_for_sw, desc="Sliding window eval"):
    # Load RGB image
    img_pil = Image.open(img_path).convert("RGB")
    orig_w, orig_h = img_pil.size

    # Rasterize SVG mask at original image size
    raw_mask = rasterizer.rasterize(svg_path, target_width=orig_w, target_height=orig_h, sparse=True)

    # Remap using the same LUT as the dataset
    max_sid = max(max(present_mapping.keys()), 255)
    lut = np.zeros(max_sid + 1, dtype=np.int64)
    for sid, cid in present_mapping.items():
        if sid <= max_sid:
            lut[sid] = cid
    lut[255] = 255
    class_mask = lut[raw_mask]

    # Resize image + mask for sliding window (avoids OOM on 6K-12K pixel images)
    image_resized, mask_resized, scale = resize_for_sliding_window(img_pil, class_mask, SW_MAX_DIM)

    pred = sliding_window_predict_rgb(
        model, image_resized, NUM_LABELS, CROP_SIZE, STRIDE, device,
    )
    pred_h, pred_w = pred.shape
    mask_resized = mask_resized[:pred_h, :pred_w]

    # Only count labeled pixels (exclude 255)
    valid = mask_resized != 255
    if valid.sum() > 0:
        all_sw_preds.append(pred[valid].ravel())
        all_sw_labels.append(mask_resized[valid].ravel())

if all_sw_preds:
    preds_flat = np.concatenate(all_sw_preds)
    labels_flat = np.concatenate(all_sw_labels)

    sw_accuracy = float((preds_flat == labels_flat).sum()) / len(labels_flat)

    sw_class_ious = {}
    for cls in range(NUM_LABELS):
        label_mask = labels_flat == cls
        if label_mask.sum() == 0:
            continue
        pred_mask = preds_flat == cls
        intersection = (pred_mask & label_mask).sum()
        union = (pred_mask | label_mask).sum()
        sw_class_ious[cls] = float(intersection) / float(union) if union > 0 else 0.0

    sw_miou = float(np.mean(list(sw_class_ious.values()))) if sw_class_ious else 0.0
    sw_valid = len(sw_class_ious)

    print(f"\n{'='*65}")
    print(f"EVALUATION RESULTS")
    print(f"{'='*65}")
    print(f"  Center-crop mIoU:        {current_miou:.1%}")
    print(f"  Center-crop accuracy:    {current_acc:.1%}")
    print(f"  Sliding window mIoU:     {sw_miou:.1%} ({len(val_pairs_for_sw)} images, resized to max {SW_MAX_DIM}px)")
    print(f"  Sliding window accuracy: {sw_accuracy:.1%}")
    print(f"  SW valid classes:        {sw_valid}")
    print(f"  mIoU delta (SW - CC):    {sw_miou - current_miou:+.1%}")

    mlflow.log_metrics({
        "sw_mean_iou": sw_miou,
        "sw_overall_accuracy": sw_accuracy,
        "sw_valid_classes": sw_valid,
        "sw_num_images": len(val_pairs_for_sw),
        "sw_max_dim": SW_MAX_DIM,
    })
else:
    print("No valid sliding window predictions (all masks empty?)")

model = model.cpu()
torch.cuda.empty_cache()

In [0]:
# Cell 7 — Save final model + log to MLflow + close run

import os
from transformers import AutoImageProcessor

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

# 1. Set id2label/label2id
model.config.id2label = {i: name for i, name in enumerate(class_names)}
model.config.label2id = {name: i for i, name in enumerate(class_names)}
print(f"Set id2label/label2id: {NUM_LABELS} classes")

# 2. Save model to DBFS
trainer.save_model(FINAL_MODEL_DIR)
print(f"Model saved to: {FINAL_MODEL_DIR}")

# 3. Save image processor
processor = AutoImageProcessor.from_pretrained(model_path)
processor.save_pretrained(FINAL_MODEL_DIR)
print(f"Image processor saved to: {FINAL_MODEL_DIR}")

# 4. Save present_mapping for later inference
import json
mapping_path = os.path.join(FINAL_MODEL_DIR, "present_mapping.json")
with open(mapping_path, "w") as f:
    json.dump({str(k): v for k, v in present_mapping.items()}, f)
print(f"Present mapping saved to: {mapping_path}")

# 5. Log to MLflow — raw file artifacts
mlflow.log_artifacts(FINAL_MODEL_DIR, artifact_path="model")
print("Model artifacts logged to MLflow under 'model/'")

# 6. Close MLflow run
mlflow.end_run()
print(f"\nMLflow run closed. Final model: {FINAL_MODEL_DIR}")